In [14]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import urllib.parse

In [15]:
#config
username = 'root'
password = '123@123'
safe_password = urllib.parse.quote_plus(password)
host = 'localhost'
port = '3306'
database_dwh = 'coffeeshop_dwh'
database_raw = 'coffeeshopdata'

In [16]:
#========================================================================
# 1. EXTRACT: Load data from raw database
#========================================================================

In [17]:
engine = create_engine(f"mysql+pymysql://{username}:{safe_password}@{host}:{port}/{database_raw}")

query = "SELECT * FROM raw_sales_data"
df_raw = pd.read_sql(query, engine)
df_raw.head()

,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail
0,1,2023-01-01,0 days 07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg
1,2,2023-01-01,0 days 07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
2,3,2023-01-01,0 days 07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg
3,4,2023-01-01,0 days 07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm
4,5,2023-01-01,0 days 07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg


In [18]:
#========================================================================
# 2. TRANSFORM: data processing
#========================================================================

In [19]:
df_clean = df_raw.copy()
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 149116 entries, 0 to 149115
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype          
---  ------            --------------   -----          
 0   transaction_id    149116 non-null  int64          
 1   transaction_date  149116 non-null  object         
 2   transaction_time  149116 non-null  timedelta64[us]
 3   transaction_qty   149116 non-null  int64          
 4   store_id          149116 non-null  int64          
 5   store_location    149116 non-null  str            
 6   product_id        149116 non-null  int64          
 7   unit_price        149116 non-null  float64        
 8   product_category  149116 non-null  str            
 9   product_type      149116 non-null  str            
 10  product_detail    149116 non-null  str            
dtypes: float64(1), int64(4), object(1), str(4), timedelta64[us](1)
memory usage: 12.5+ MB


In [20]:
df_clean.columns = df_clean.columns.str.strip().str.replace(' ', '_').str.capitalize()
df_clean['Transaction_time'] = df_clean['Transaction_time'].astype(str).str.split().str[-1]
df_clean.head()

,Transaction_id,Transaction_date,Transaction_time,Transaction_qty,Store_id,Store_location,Product_id,Unit_price,Product_category,Product_type,Product_detail
0,1,2023-01-01,07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg
1,2,2023-01-01,07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
2,3,2023-01-01,07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg
3,4,2023-01-01,07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm
4,5,2023-01-01,07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg


In [21]:
#========================================================================
# 3. LOAD: Load data to Data Warehouse
#========================================================================

In [22]:
print("Loading data to Data Warehouse...")

dwh_engine = create_engine(f"mysql+pymysql://{username}:{safe_password}@{host}:{port}/{database_dwh}")

try: 
    df_clean.to_sql(name='fact_sales', con=dwh_engine, if_exists='append', index=False)
    print("Data loaded successfully to Data Warehouse!")
except Exception as e:
    print("Error loading data to Data Warehouse:", e)

Loading data to Data Warehouse...
Data loaded successfully to Data Warehouse!
